# 03 — Modèles classiques

Référence naïve → régression linéaire → forêt aléatoire / XGBoost, sur les variables mobiles.

**Validation sans fuite** : `GroupKFold` par moteur. Le jeu de test officiel n'est utilisé qu'une fois, à la fin.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
SUBSET = "FD001"   # puis "FD004"


In [ ]:
import time
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from src.data import prepare
from src.features import rolling_features, feature_columns, last_cycle
from src.evaluate import rmse, evaluate, error_by_rul_bucket, results_table

CAP = 125
train, test, rul_test, sensors = prepare(SUBSET, cap=CAP)
train_f = rolling_features(train, sensors, 10)
test_f = rolling_features(test, sensors, 10)
cols = feature_columns(sensors)
X, y, groups = train_f[cols], train_f["RUL"], train_f["unit"]
cv = GroupKFold(n_splits=5)

## Choix du plafond de RUL (validation croisée par moteur)

In [ ]:
for cap in (120, 125, 130):
    tr, _, _, s = prepare(SUBSET, cap=cap)
    f = rolling_features(tr, s, 10)
    p = cross_val_predict(RandomForestRegressor(100, min_samples_leaf=5, n_jobs=-1, random_state=0),
                          f[feature_columns(s)], f["RUL"], groups=f["unit"], cv=cv)
    print(f"plafond {cap} : RMSE validation {rmse(f['RUL'], p):.2f}")

## Validation croisée des modèles

In [ ]:
models = {
    "Régression linéaire": LinearRegression(),
    "Forêt aléatoire": RandomForestRegressor(300, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "XGBoost": XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8,
                            colsample_bytree=0.8, random_state=0),
}
cv_rows = [{"modèle": "Référence naïve", "RMSE validation": rmse(y, np.full(len(y), y.mean()))}]
for name, m in models.items():
    p = cross_val_predict(m, X, y, groups=groups, cv=cv)
    cv_rows.append({"modèle": name, "RMSE validation": rmse(y, p)})
pd.DataFrame(cv_rows).set_index("modèle").round(2)

## Évaluation finale sur le jeu de test officiel (une seule fois)

On prédit la RUL au dernier cycle observé de chaque moteur de test.

In [ ]:
X_last = last_cycle(test_f)[cols]
y_true = rul_test.loc[X_last.index].clip(upper=CAP)   # RUL réelle plafonnée comme la cible
rows, fitted = [evaluate(y_true, np.full(len(y_true), y.mean()), "Référence naïve")], {}
for name, m in models.items():
    t0 = time.time(); m.fit(X, y); secs = time.time() - t0
    pred = np.clip(m.predict(X_last), 0, None)
    rows.append({**evaluate(y_true, pred, name), "temps (s)": secs})
    fitted[name] = pred
res = results_table(rows)
res.to_csv(ROOT / "reports" / f"resultats_classiques_{SUBSET}.csv")
res

In [ ]:
best = res.drop("Référence naïve")["RMSE"].idxmin()
pred = fitted[best]
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_true, pred, s=12, alpha=0.7); ax.plot([0, CAP], [0, CAP], "k--", lw=1)
ax.set(xlabel="RUL réelle", ylabel="RUL prédite", title=f"{best} — test {SUBSET}")
plt.tight_layout(); plt.savefig(FIG / f"pred_vs_reel_{best.replace(' ', '_')}_{SUBSET}.png", dpi=150); plt.show()
error_by_rul_bucket(y_true, pred)

## Interprétation : importance des capteurs (SHAP)

In [ ]:
import shap
xgb = models["XGBoost"]
sample = X.sample(2000, random_state=0)
shap_values = shap.TreeExplainer(xgb).shap_values(sample)
shap.summary_plot(shap_values, sample, max_display=15, show=False)
plt.tight_layout(); plt.savefig(FIG / f"shap_{SUBSET}.png", dpi=150); plt.show()